# Multi-Hospital Lab Results: Data Cleaning Pipeline

## 1. Environment Setup & Data Import
To begin our programmatic data cleaning pipeline, we import the necessary Python libraries (Pandas and NumPy) for data manipulation and load our raw, unstructured CSV dataset into a Pandas DataFrame.

In [1]:
import pandas as pd
import numpy as np

In [2]:
med_res = pd.read_csv(r"C:\Users\User\Downloads\multi_hospital_lab_results.csv")

## 2. Initial Data Inspection
Before applying any transformations, we inspect the first few rows and the underlying schema of the dataset to identify immediate data quality issues.

**Key Initial Observations:**
*   **Sentinel Values:** Placeholder values (`-999`) are present in the numeric `test_value` column.
*   **Inconsistent Naming:** The `test_name` column contains non-standardized phrasing (e.g., 'Fasting Glucose' vs. 'Glucose').
*   **Data Type Errors:** `patient_id` is loaded as an integer (`int64`), which should be treated as a categorical string. `collection_date` is currently formatted as a text object rather than a datetime object.

In [3]:
med_res.head()

,hospital,patient_id,test_name,test_value,unit,reference_range,collection_date
0,Hospital_A,1966,Fasting Glucose,8.56,mmol/L,70-99,2025-08-15
1,Hospital_C,1826,Glucose,-999,mmol/L,70-99,2024-07-04
2,Hospital_A,1606,Cholesterol,6.45,mmol/L,125-200,2024-11-13
3,Hospital_B,1277,Fasting Glucose,-999,mmol/L,3.9-5.5,2024-04-29
4,Hospital_A,1841,Serum Cholesterol,5.61,mmol/L,<200,2024-08-05


In [4]:
med_res.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1500 entries, 0 to 1499
Data columns (total 7 columns):
 #   Column           Non-Null Count  Dtype 
---  ------           --------------  ----- 
 0   hospital         1500 non-null   object
 1   patient_id       1500 non-null   int64 
 2   test_name        1500 non-null   object
 3   test_value       1461 non-null   object
 4   unit             1500 non-null   object
 5   reference_range  1500 non-null   object
 6   collection_date  1500 non-null   object
dtypes: int64(1), object(6)
memory usage: 82.2+ KB


## 3. Data Type Corrections
Based on our initial inspection, we apply immediate data type corrections to prevent aggregation errors later in our analysis. We convert `patient_id` to a standard string format and cast `collection_date` to a proper Python datetime object.

In [5]:
med_res['patient_id'] = med_res['patient_id'].astype(str)

In [6]:
med_res['collection_date'] = pd.to_datetime(med_res['collection_date'])

## 4. Standardizing Clinical Terminology
The `test_name` column contains various colloquial and abbreviated names for standard lab tests across different hospitals. We first extract an array of all unique test names to map out the inconsistencies. Then, we define a custom logical function to unify these variations into standard clinical terminology.

In [7]:
med_res['test_name'].unique()

array(['Fasting Glucose', 'Glucose', 'Cholesterol', 'Serum Cholesterol',
       'Glycated Hemoglobin', 'Hemoglobin A1c', 'Blood Sugar',
       'Total Chol', 'HbA1c'], dtype=object)

In [8]:
def standardized_name(test):
    if test == 'Fasting Glucose':
        return 'Glucose'
    elif test == 'Blood Sugar':
        return 'Glucose'
    elif test == 'Serum Cholesterol':
        return 'Cholesterol'
    elif test == 'Total Chol':
        return 'Cholesterol'
    elif test == 'Glycated Hemoglobin':
        return 'Hemoglobin A1c'
    elif test == 'HbA1c':
        return 'Hemoglobin A1c'
    else:
        return test

med_res['test_name'] = med_res['test_name'].apply(standardized_name)

In [9]:
med_res['test_name'].unique()

array(['Glucose', 'Cholesterol', 'Hemoglobin A1c'], dtype=object)

## 5. Handling Missing and Invalid Data
The `test_value` column contains mixed data types (text artifacts) and placeholder values (`-999`). We force the column to a numeric data type, which coerces hidden text into null values (`NaN`). Next, we target and replace the `-999` sentinel values with `NaN` to ensure accurate statistical calculations later.

In [10]:
med_res['test_value'].isnull().sum()

np.int64(39)

In [11]:
med_res['test_value'] = pd.to_numeric(med_res['test_value'], errors = 'coerce')

In [12]:
med_res['test_value'] = med_res['test_value'].replace(-999, np.nan)

In [13]:
med_res['test_value'].isnull().sum()

np.int64(101)

## 6. Unit Conversion and Standardization
To ensure all clinical values are mathematically comparable, we must standardize the measurement units. We apply a custom logical function to identify rows recorded in `mg/dL` and convert them to the standard `mmol/L` by dividing by their respective molecular weights (18 for Glucose, 38.67 for Cholesterol). Finally, we update the `unit` column to reflect these changes.

In [14]:
def standardized_value(convert):
    name = convert['test_name']
    value = convert['test_value']
    unit = convert['unit']

    if unit == 'mg/dL':
        if name == 'Glucose':
            return round(value/18, 2)
        elif name == 'Cholesterol':
            return round(value/38.67, 2)
        else:
            return value
    else:
        return value

med_res['test_value'] = med_res.apply(standardized_value, axis = 1)

In [15]:
med_res['unit'] = med_res['unit'].replace('mg/dL', 'mmol/L')

## 7. Extracting Reference Ranges
The `reference_range` column originally contained inconsistent string formats. We standardize these ranges using a custom function based on the test type. Then, to make the ranges mathematically queryable, we split the string at the hyphen (`-`) into two distinct columns (`min_range` and `max_range`) and convert them to float data types.

In [16]:
def reference(rang):
    name = rang['test_name']
    value = rang['reference_range']

    if name == 'Glucose':
        return '3.9-5.5'
    elif name == 'Cholesterol':
        return '3.2-5.2'
    elif name == 'Hemoglobin A1c':
        return '4.0-5.6'
    else:
        return value

med_res['reference_range'] = med_res.apply(reference, axis = 1)

In [17]:
med_res[['min_range', 'max_range']] = med_res['reference_range'].str.split('-', expand = True)

In [18]:
med_res['min_range'] = pd.to_numeric(med_res['min_range'])

In [19]:
med_res['max_range'] = pd.to_numeric(med_res['max_range'])

## 8. Finalizing the Schema
With the new minimum and maximum range columns created, the original `reference_range` string column is now obsolete. We declare a new, logical column order and overwrite the DataFrame, which seamlessly rearranges our data and drops the unneeded column in one step.

In [20]:
new_column_order = [
    'hospital', 
    'patient_id', 
    'test_name', 
    'test_value', 
    'unit', 
    'min_range',
    'max_range',
    'collection_date'
]
med_res = med_res[new_column_order]

## 9. Verification and Data Export
As a final step, we inspect the DataFrame's `.info()` to verify that all columns possess the correct data types (strings as objects, numeric values as floats, and dates as datetime objects). After confirming the final layout with `.head()`, we export the cleaned dataset to a new CSV file, ready for downstream analysis and visualization.

In [21]:
med_res.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1500 entries, 0 to 1499
Data columns (total 8 columns):
 #   Column           Non-Null Count  Dtype         
---  ------           --------------  -----         
 0   hospital         1500 non-null   object        
 1   patient_id       1500 non-null   object        
 2   test_name        1500 non-null   object        
 3   test_value       1399 non-null   float64       
 4   unit             1500 non-null   object        
 5   min_range        1500 non-null   float64       
 6   max_range        1500 non-null   float64       
 7   collection_date  1500 non-null   datetime64[ns]
dtypes: datetime64[ns](1), float64(3), object(4)
memory usage: 93.9+ KB


In [22]:
med_res.head()

,hospital,patient_id,test_name,test_value,unit,min_range,max_range,collection_date
0,Hospital_A,1966,Glucose,8.56,mmol/L,3.9,5.5,2025-08-15
1,Hospital_C,1826,Glucose,NaN,mmol/L,3.9,5.5,2024-07-04
2,Hospital_A,1606,Cholesterol,6.45,mmol/L,3.2,5.2,2024-11-13
3,Hospital_B,1277,Glucose,NaN,mmol/L,3.9,5.5,2024-04-29
4,Hospital_A,1841,Cholesterol,5.61,mmol/L,3.2,5.2,2024-08-05


In [23]:
med_res.to_csv('cleaned_dataset_pandas.csv', index=False)